# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
import numpy as np
import optuna

from Challenge.paths import load_cv_folds, generate_submission, MODEL_DIR
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(k=5)

# **Hyperparameter search**

In [5]:
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender

optimizer = ModelOptimizer("NMF")

STUDY_NAME = NMFRecommender.RECOMMENDER_NAME + "_v1"

In [6]:
URM_train, URM_validation = folds[0] # Too slow to test on all folds

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "num_factors": optuna_trial.suggest_int("num_factors", 50, 200),
        "l1_ratio": optuna_trial.suggest_float("l1_ratio", 0.0, 1.0),
        "init_type": optuna_trial.suggest_categorical("init_type", ["random", "nndsvda"]),
        "solver_beta_loss": optuna_trial.suggest_categorical("solver_beta_loss", ["multiplicative_update:frobenius", "multiplicative_update:kullback-leibler", "coordinate_descent:frobenius"]),
        "verbose": False,
        "random_seed": 42
    }
    

    # Train the recommender
    recommender_instance = NMFRecommender(URM_train)
    recommender_instance.fit(**params)
        
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        
    # Log folds performance
    optimizer.log_folds([score], params)

    # Return the mean CV score for the fully completed trial
    return score

In [7]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-28 20:14:49,542] A new study created in RDB with name: NMFRecommender_v1


  0%|          | 0/100 [00:00<?, ?it/s]

NMFRecommender: Computing NMF decomposition...
NMFRecommender: Computing NMF decomposition... done in 7.02 min


Eval Batches: 100%|██████████| 28/28 [00:12<00:00,  2.20it/s]


[I 2025-11-28 20:22:03,817] Trial 0 finished with value: 0.19995720920887552 and parameters: {'num_factors': 80, 'l1_ratio': 0.9599134004009405, 'init_type': 'random', 'solver_beta_loss': 'multiplicative_update:kullback-leibler'}. Best is trial 0 with value: 0.19995720920887552.
NMFRecommender: Computing NMF decomposition...
NMFRecommender: Computing NMF decomposition... done in 1.95 min


...

Eval Batches: 100%|██████████| 28/28 [00:21<00:00,  1.31it/s]


[I 2025-11-28 21:16:37,453] Trial 10 finished with value: 0.19366162567323322 and parameters: {'num_factors': 155, 'l1_ratio': 0.8940239992519566, 'init_type': 'random', 'solver_beta_loss': 'multiplicative_update:kullback-leibler'}. Best is trial 0 with value: 0.19995720920887552.
NMFRecommender: Computing NMF decomposition...
NMFRecommender: Computing NMF decomposition... done in 18.45 min


Eval Batches: 100%|██████████| 28/28 [00:28<00:00,  1.00s/it]


[I 2025-11-28 21:35:32,302] Trial 11 finished with value: 0.19117822110815763 and parameters: {'num_factors': 170, 'l1_ratio': 0.9991067196095147, 'init_type': 'random', 'solver_beta_loss': 'multiplicative_update:kullback-leibler'}. Best is trial 0 with value: 0.19995720920887552.
NMFRecommender: Computing NMF decomposition...
[W 2025-11-28 21:36:05,727] Trial 12 failed with parameters: {'num_factors': 151, 'l1_ratio': 0.9702295689303904, 'init_type': 'random', 'solver_beta_loss': 'multiplicative_update:kullback-leibler'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/luigi/.venvs/recsys/lib/python3.13/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_18225/2528373942.py", line 16, in objective_function
    recommender_instance.fit(**params)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/home/luigi/RecSys/Recommenders/MatrixFactorization/NMFRecommender.py", l

KeyboardInterrupt: 

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [ ]:
bp = optimizer.get_best_params(STUDY_NAME)

def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "topK": optuna_trial.suggest_int("topK", max(10, bp["topK"]-50), bp["topK"]+50),
        "shrink": optuna_trial.suggest_int("shrink", max(0, bp["shrink"]-50), bp["shrink"]+50),
        "similarity": "cosine",
        "normalize": True
    }
    
    validation_scores = []
    for URM_train, URM_validation in folds:        
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {len(validation_scores)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(np.mean(validation_scores), len(validation_scores))

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            raise optuna.TrialPruned()
        
        # Log fold performance
        optimizer.log_fold_performance(len(validation_scores), score)

    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- ADD HERE

# **Train Model with best hyperparameter**

In [ ]:
# Train final model on train + validation with best hyperparameters
URM_train, URM_validation = folds[0]
URM_all = URM_train + URM_validation

bp = optimizer.get_best_params(STUDY_NAME+"_refined")

recommender = ItemKNNCFRecommender(URM_all)
recommender.fit(
    topK=bp["topK"],
    shrink=bp["shrink"],
    similarity="cosine",
    normalize=True
)

# Save the trained model
recommender.save_model(MODEL_DIR)

In [ ]:
generate_submission(recommender, STUDY_NAME+".csv")